# LAB 4#

## JUAN PABLO PEREZ LOPEZ ##


In [ ]:
from spark_utils import SparkUtils


su = SparkUtils("spark://spark-master:7077", 'LAB 4')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/28 17:50:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
from pyspark.sql.functions import from_json
agencies_json_schema = SparkUtils.generate_schema([('agency_id','int'),('json_col','string')])
agencies_df_json = su._spark.read.option("header", "true").schema(agencies_json_schema).csv("/opt/spark/work-dir/data/car_service/agencies.csv")


# Define the schema of the JSON object
agencies_schema = SparkUtils.generate_schema([("agency_name", "string"),
                                            ("city", "string")])
df_agencies_parsed = agencies_df_json.withColumn("parsed", from_json(agencies_df_json.json_col, agencies_schema))
df_agencies_parsed.printSchema()
df_agencies_parsed.show(truncate=False)




root
 |-- agency_id: integer (nullable = true)
 |-- json_col: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- agency_name: string (nullable = true)
 |    |-- city: string (nullable = true)

+---------+-----------------------------------------------------+--------------------------+
|agency_id|json_col                                             |parsed                    |
+---------+-----------------------------------------------------+--------------------------+
|1        |{'agency_name': 'NYC Rentals', 'city': 'New York'}   |{NYC Rentals, New York}   |
|2        |{'agency_name': 'LA Car Rental', 'city': 'Londres'}  |{LA Car Rental, Londres}  |
|3        |{'agency_name': 'Zapopan Auto', 'city': 'Zapopan'}   |{Zapopan Auto, Zapopan}   |
|4        |{'agency_name': 'SF Cars', 'city': 'San Francisco'}  |{SF Cars, San Francisco}  |
|5        |{'agency_name': 'Mexico Cars', 'city': 'Mexico City'}|{Mexico Cars, Mexico City}|
+---------+------------------------------

In [9]:
from pyspark.sql.functions import col
df_agencies=df_agencies_parsed.select(col('agency_id'),col('parsed.agency_name'),col('parsed.city'))

df_agencies.show(truncate=False)

+---------+-------------+-------------+
|agency_id|agency_name  |city         |
+---------+-------------+-------------+
|1        |NYC Rentals  |New York     |
|2        |LA Car Rental|Londres      |
|3        |Zapopan Auto |Zapopan      |
|4        |SF Cars      |San Francisco|
|5        |Mexico Cars  |Mexico City  |
+---------+-------------+-------------+



In [11]:

brands_json_schema = SparkUtils.generate_schema([('brand_id','int'),('json_col','string')])
brands_df_json = su._spark.read.option("header", "true").schema(brands_json_schema).csv("/opt/spark/work-dir/data/car_service/brands.csv")


# Define the schema of the JSON object
brands_schema = SparkUtils.generate_schema([("brand_name", "string"),
                                            ("country", "string")])
df_brands_parsed = brands_df_json.withColumn("parsed", from_json(brands_df_json.json_col, brands_schema))
df_brands_parsed.printSchema()
df_brands_parsed.show(truncate=False)




root
 |-- brand_id: integer (nullable = true)
 |-- json_col: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- brand_name: string (nullable = true)
 |    |-- country: string (nullable = true)

+--------+-----------------------------------------------------+------------------------+
|brand_id|json_col                                             |parsed                  |
+--------+-----------------------------------------------------+------------------------+
|1       |{'brand_name': 'Mercedes-Benz', 'country': 'Germany'}|{Mercedes-Benz, Germany}|
|2       |{'brand_name': 'BMW', 'country': 'Germany'}          |{BMW, Germany}          |
|3       |{'brand_name': 'Audi', 'country': 'Germany'}         |{Audi, Germany}         |
|4       |{'brand_name': 'Ford', 'country': 'US'}              |{Ford, US}              |
|5       |{'brand_name': 'BYD', 'country': 'China'}            |{BYD, China}            |
|6       |{'brand_name': 'Honda', 'country': 'Japan'}          |

In [15]:
df_brands=df_brands_parsed.select(col('brand_id'),col('parsed.brand_name'),col('parsed.country'))

df_brands.show(truncate=False)

+--------+-------------+-------+
|brand_id|brand_name   |country|
+--------+-------------+-------+
|1       |Mercedes-Benz|Germany|
|2       |BMW          |Germany|
|3       |Audi         |Germany|
|4       |Ford         |US     |
|5       |BYD          |China  |
|6       |Honda        |Japan  |
|7       |Toyota       |Japan  |
+--------+-------------+-------+



In [19]:
cars_json_schema = SparkUtils.generate_schema([('car_id','int'),('json_col','string')])
cars_df_json = su._spark.read.option("header", "true").schema(cars_json_schema).csv("/opt/spark/work-dir/data/car_service/cars.csv")


# Define the schema of the JSON object
cars_schema = SparkUtils.generate_schema([("car_name", "string"),
                                            ("brand_id",'int'),
                                            ("price_per_day", "int")])
df_cars_parsed = cars_df_json.withColumn("parsed", from_json(cars_df_json.json_col, cars_schema))
df_cars_parsed.printSchema()
df_cars_parsed.show(5,truncate=False)


df_cars=df_cars_parsed.select(col('car_id'),col('parsed.car_name'),col('parsed.brand_id'),col('parsed.price_per_day'))
df_cars.show(5)

root
 |-- car_id: integer (nullable = true)
 |-- json_col: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- car_name: string (nullable = true)
 |    |-- brand_id: integer (nullable = true)
 |    |-- price_per_day: integer (nullable = true)

+------+---------------------------------------------------------------------------+--------------------------------+
|car_id|json_col                                                                   |parsed                          |
+------+---------------------------------------------------------------------------+--------------------------------+
|1     |{'car_name': 'Chang-Fisher Model 7', 'brand_id': 5, 'price_per_day': 139}  |{Chang-Fisher Model 7, 5, 139}  |
|2     |{'car_name': 'Sheppard-Tucker Model 4', 'brand_id': 6, 'price_per_day': 70}|{Sheppard-Tucker Model 4, 6, 70}|
|3     |{'car_name': 'Faulkner-Howard Model 5', 'brand_id': 3, 'price_per_day': 53}|{Faulkner-Howard Model 5, 3, 53}|
|4     |{'car_name': 'Wagne

In [ ]:
customers_json_schema = SparkUtils.generate_schema([('customer_id','int'),('json_col','string')])
customers_df_json = su._spark.read.option("header", "true").schema(customers_json_schema).csv("/opt/spark/work-dir/data/car_service/customers.csv")


# Define the schema of the JSON object
customers_schema = SparkUtils.generate_schema([("customer_name", "string"),
                                            ("city",'string'),
                                            ("age", "int")])
df_customers_parsed = customers_df_json.withColumn("parsed", from_json(customers_df_json.json_col, customers_schema))
df_customers_parsed.printSchema()
df_customers_parsed.show(5,truncate=False)


df_customers=df_customers_parsed.select(col('customer_id'),col('parsed.customer_name'),col('parsed.city'),col('parsed.age'))
df_customers.show(5)

root
 |-- customer_id: integer (nullable = true)
 |-- json_col: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- customer_name: string (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- age: integer (nullable = true)

+-----------+---------------------------------------------------------------------+---------------------------------+
|customer_id|json_col                                                             |parsed                           |
+-----------+---------------------------------------------------------------------+---------------------------------+
|1          |{'customer_name': 'Tiffany Riley', 'city': 'Monterrey', 'age': 32}   |{Tiffany Riley, Monterrey, 32}   |
|2          |{'customer_name': 'Matthew Davies', 'city': 'Monterrey', 'age': 36}  |{Matthew Davies, Monterrey, 36}  |
|3          |{'customer_name': 'Rebecca Miller', 'city': 'Mexico City', 'age': 30}|{Rebecca Miller, Mexico City, 30}|
|4          |{'customer_name': '

In [25]:
rentals_json_schema = SparkUtils.generate_schema([('rental_id','int'),('json_col','string')])
rentals_df_json = su._spark.read.option("header", "true").schema(rentals_json_schema).csv("/opt/spark/work-dir/data/car_service/rentals/")


# Define the schema of the JSON object
rentals_schema = SparkUtils.generate_schema([("car_id", "int"),
                                            ("customer_id",'int'),
                                            ("agency_id", "int")])
df_rentals_parsed = rentals_df_json.withColumn("parsed", from_json(rentals_df_json.json_col, rentals_schema))
df_rentals_parsed.printSchema()
df_rentals_parsed.show(5,truncate=False)


df_rentals=df_rentals_parsed.select(col('rental_id'),col('parsed.car_id'),col('parsed.customer_id'),col('parsed.agency_id'))
df_rentals.show(5)

root
 |-- rental_id: integer (nullable = true)
 |-- json_col: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- car_id: integer (nullable = true)
 |    |-- customer_id: integer (nullable = true)
 |    |-- agency_id: integer (nullable = true)

+---------+--------------------------------------------------+------------+
|rental_id|json_col                                          |parsed      |
+---------+--------------------------------------------------+------------+
|11891    |{'car_id': 21, 'customer_id': 71, 'agency_id': 1} |{21, 71, 1} |
|11892    |{'car_id': 11, 'customer_id': 52, 'agency_id': 2} |{11, 52, 2} |
|11893    |{'car_id': 22, 'customer_id': 116, 'agency_id': 4}|{22, 116, 4}|
|11894    |{'car_id': 5, 'customer_id': 107, 'agency_id': 1} |{5, 107, 1} |
|11895    |{'car_id': 4, 'customer_id': 53, 'agency_id': 4}  |{4, 53, 4}  |
+---------+--------------------------------------------------+------------+
only showing top 5 rows
+---------+------+--------

In [27]:
#limpiamos los demas df para dejarlos como dimension
agencies_dim=df_agencies.select('agency_id','agency_name')

agencies_dim.show()

+---------+-------------+
|agency_id|  agency_name|
+---------+-------------+
|        1|  NYC Rentals|
|        2|LA Car Rental|
|        3| Zapopan Auto|
|        4|      SF Cars|
|        5|  Mexico Cars|
+---------+-------------+



In [28]:
cars_dim=df_cars.select('car_id','car_name')
cars_dim.show(5)

+------+--------------------+
|car_id|            car_name|
+------+--------------------+
|     1|Chang-Fisher Model 7|
|     2|Sheppard-Tucker M...|
|     3|Faulkner-Howard M...|
|     4|  Wagner LLC Model 1|
|     5|  Campos PLC Model 4|
+------+--------------------+
only showing top 5 rows


In [29]:
customer_dim=df_customers.select('customer_id','customer_name')
customer_dim.show(5)

+-----------+--------------+
|customer_id| customer_name|
+-----------+--------------+
|          1| Tiffany Riley|
|          2|Matthew Davies|
|          3|Rebecca Miller|
|          4| Katelyn Mccoy|
|          5|   Dana Dennis|
+-----------+--------------+
only showing top 5 rows


26/03/01 01:06:13 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.4: worker lost: Not receiving heartbeat for 60 seconds
26/03/01 01:06:13 ERROR TaskSchedulerImpl: Lost executor 2 on 172.18.0.3: worker lost: Not receiving heartbeat for 60 seconds
26/03/01 01:06:13 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.5: worker lost: Not receiving heartbeat for 60 seconds


In [59]:
#we make joins to create last df

df_final=(df_rentals.join(cars_dim,on='car_id',how='left')
                    .join(agencies_dim,on='agency_id',how='left')
                    .join(customer_dim, on='customer_id',how='left'))


result=df_final
result=result.select('rental_id','car_name','agency_name','customer_name')

result.show(20, truncate=False)



+---------+-----------------------------------+-------------+---------------+
|rental_id|car_name                           |agency_name  |customer_name  |
+---------+-----------------------------------+-------------+---------------+
|11891    |Wallace-Carlson Model 9            |NYC Rentals  |Margaret Jones |
|11892    |Grimes-Green Model 8               |LA Car Rental|Albert Williams|
|11893    |Stewart-Allen Model 5              |SF Cars      |Caleb Fleming  |
|11894    |Campos PLC Model 4                 |NYC Rentals  |Andrew Butler  |
|11895    |Wagner LLC Model 1                 |SF Cars      |Kristin Potts  |
|11896    |Jones, Jefferson and Rivera Model 7|LA Car Rental|Jeremy Parks   |
|11897    |Lopez and Sons Model 9             |Zapopan Auto |Terry Wells    |
|11898    |Salazar Ltd Model 8                |SF Cars      |Marc Williams  |
|11899    |Villanueva PLC Model 7             |LA Car Rental|Danny Williams |
|11900    |Faulkner-Howard Model 5            |SF Cars      |Eri

In [60]:
su._spark.stop()